# 01 · Validación de datos

Etapa 1 del orden obligatorio. `run_data_validation` lee los CSV de M5, valida el calendario, calcula el lanzamiento por tienda, enmascara (NaN) prelanzamiento y datos inválidos, y agrega por artículo.

Flujo de este notebook: ejecutar **sin guardar**, auditar las entradas y el efecto de cada máscara, y recién al final guardar en `data/interim/` con `save_data_validation`. En un proceso automático se usa `persist=True`.

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

versión de la config: v1


In [ ]:
cfg

{'version': 'v1',
 'data': {'raw_dir': 'data/raw',
  'sales_file': 'sales_train_evaluation.csv',
  'last_day': 1941},
 'split': {'history_days': [1, 121],
  'years': {'Y1': [122, 485],
   'Y2': [486, 849],
   'Y3': [850, 1213],
   'Y4': [1214, 1577]},
  'test_days': [1578, 1941],
  'test_origins': 13,
  'horizon': 28},
 'segmentation': {'usage_period': 'computation_set',
  'abc_cuts': [0.8, 0.95],
  'xyz_thresholds': [10.0, 25.0],
  'cv_aggregation': 'block28',
  'cv_min_blocks': 2},
 'stability': {'segment_candidates': ['quarter', 'four_month'],
  'segment_criterion': 'max_mean_abc_agreement',
  'min_active_days_year': 364,
  'agreement_min': 0.8,
  'psi_bands': [0.1, 0.25],
  'psi_epsilon': 0.0001,
  'stability_rule': 'abc_all',
  'stability_rule_criterion': 'fixed'},
 'treatment_map': {'AX': 'regular_replenishment',
  'AY': 'periodic_review_replenishment',
  'AZ': 'promotion_priority'},
 'forecast': {'selection_level': 'item',
  'subset_size': 'all',
  'backtest_folds': 4,
  'snaive

## 1. Ejecutar sin guardar

`persist=False` (por defecto) no escribe nada. La salida trae los datos crudos y los enmascarados para compararlos.

In [ ]:
from src.data import (
    MASK_REASONS,
    audit_series,
    mask_summary,
    run_data_validation,
    save_data_validation,
)

out = run_data_validation(cfg)
pd.Series({k: getattr(v, "shape", type(v).__name__) for k, v in out.items()}, name="forma / tipo")

calendar               (1969, 14)
week_of_day               (1941,)
prices_raw           (6841121, 5)
series_index           (30490, 3)
units_raw           (30490, 1941)
prices_daily_raw    (30490, 1941)
units               (30490, 1941)
prices_daily        (30490, 1941)
mask_reason         (30490, 1941)
launch                 (30490, 4)
rejections                 (0, 5)
agg                          dict
persisted                    list
Name: forma / tipo, dtype: object

## 2. Entradas crudas

Calendario validado, precios semanales tal como se leyeron y unidades antes de enmascarar.

In [ ]:
out["calendar"][["d", "date", "wm_yr_wk"]].agg(["min", "max"])

,d,date,wm_yr_wk
min,d_1,2011-01-29,11101
max,d_999,2016-06-19,11621


In [ ]:
prices = out["prices_raw"]
print("filas de precios:", len(prices))
print("precios ≤ 0 o faltantes:", int((~(prices["sell_price"] > 0)).sum()))
prices["sell_price"].describe()

filas de precios: 6841121
precios ≤ 0 o faltantes: 0


count    6.841121e+06
mean     4.410952e+00
std      3.408814e+00
min      1.000000e-02
25%      2.180000e+00
50%      3.470000e+00
75%      5.840000e+00
max      1.073200e+02
Name: sell_price, dtype: float64

In [ ]:
raw = out["units_raw"]
finite = np.isfinite(raw)
pd.Series({
    "celdas": raw.size,
    "faltantes (NaN/inf)": int((~finite).sum()),
    "negativas": int((finite & (raw < 0)).sum()),
    "no enteras": int((finite & (raw != np.floor(raw))).sum()),
    "ceros": int((raw == 0).sum()),
}, name="unidades crudas")

celdas                 59181090
faltantes (NaN/inf)           0
negativas                     0
no enteras                    0
ceros                  40241819
Name: unidades crudas, dtype: int64

## 3. Efecto de las máscaras

Una fila por motivo (`MASK_REASONS`): celdas, series y artículos afectados, y unidades crudas que quedan en NaN. `pre_launch` no es un rechazo; `price_gap_no_sales` es solo informativo.

In [ ]:
summary = mask_summary(out["mask_reason"], out["units_raw"], out["series_index"])
summary

,code,reason,is_rejection,cells,pct_cells,series,items,units_removed
0,0,valid,False,46881677,79.217326,30490,3049,0.0
1,1,pre_launch,False,12299413,20.782674,19558,2718,0.0
2,2,sales_before_first_price,True,0,0.000000,0,0,0.0
3,3,invalid_units,True,0,0.000000,0,0,0.0
4,4,invalid_price_with_sales,True,0,0.000000,0,0,0.0
5,5,price_gap_no_sales,False,0,0.000000,0,0,0.0


Comprobaciones: cada NaN tiene motivo, cada celda válida conserva su valor crudo y el total de la cadena solo pierde las unidades enmascaradas.

In [ ]:
codes = out["mask_reason"]
units = out["units"]
assert np.array_equal(np.isnan(units), codes > 0)
valid = codes == 0
assert np.array_equal(units[valid], out["units_raw"][valid])
lost = np.nansum(np.where(codes > 0, out["units_raw"], 0), dtype=np.float64)
kept = out["agg"]["chain_total"].sum()
raw_total = np.nansum(out["units_raw"], dtype=np.float64)
assert np.isclose(kept + lost, raw_total)
print(f"unidades crudas {raw_total:,.0f} = conservadas {kept:,.0f} + enmascaradas {lost:,.0f}")

unidades crudas 66,927,173 = conservadas 66,927,173 + enmascaradas 0


## 4. Registro de rechazos

Cada rechazo debe tener motivo, fuente y fila (SC-002). `row_id` es la fila de datos en el CSV de la fuente (base 0).

In [ ]:
rejections = out["rejections"]
assert rejections[["reason", "source", "row_id"]].notna().all().all()
display(rejections.groupby(["reason", "source"]).size().rename("filas"))
rejections.head(20)

Series([], Name: filas, dtype: int64)

,reason,source,row_id,column,detail


## 5. Lanzamiento

Primera semana con precio por tienda-artículo (FR-004). Las series que nunca tienen precio quedan completas en `pre_launch`.

In [ ]:
launch = out["launch"]
print("series sin precio nunca:", int(launch["launch_d"].isna().sum()))
launch["launch_d"].describe()

series sin precio nunca: 0


count    30490.000000
mean       404.391702
std        476.005658
min          1.000000
25%          1.000000
50%        155.000000
75%        764.000000
max       1842.000000
Name: launch_d, dtype: float64

## 6. Revisión de una serie

Crudo frente a enmascarado, día a día, alrededor del lanzamiento de la serie lanzada más tarde. Cambiar `store_id`, `item_id` y el rango para revisar cualquier otra.

In [ ]:
row = launch.loc[launch["launch_d"].idxmax()]
d0 = int(row["launch_d"])
print(row["store_id"], row["item_id"], "lanzado el día", d0)
audit_series(out, row["store_id"], row["item_id"], max(1, d0 - 7), d0 + 13)

CA_1 FOODS_3_595 lanzado el día 1842


,d,date,wm_yr_wk,units_raw,price_raw,mask_reason,units,price
0,1835,2016-02-06,11602,0.0,NaN,pre_launch,NaN,NaN
1,1836,2016-02-07,11602,0.0,NaN,pre_launch,NaN,NaN
2,1837,2016-02-08,11602,0.0,NaN,pre_launch,NaN,NaN
3,1838,2016-02-09,11602,0.0,NaN,pre_launch,NaN,NaN
4,1839,2016-02-10,11602,0.0,NaN,pre_launch,NaN,NaN
5,1840,2016-02-11,11602,0.0,NaN,pre_launch,NaN,NaN
6,1841,2016-02-12,11602,0.0,NaN,pre_launch,NaN,NaN
7,1842,2016-02-13,11603,0.0,1.98,valid,0.0,1.98
8,1843,2016-02-14,11603,0.0,1.98,valid,0.0,1.98
9,1844,2016-02-15,11603,0.0,1.98,valid,0.0,1.98


## 7. Guardar

Con la revisión terminada, se guardan los intermedios que leen las etapas siguientes. Los crudos no se guardan: se releen desde `data/raw/` (FR-001).

In [ ]:
paths = save_data_validation(out)
pd.Series([str(p) for p in paths], name="archivos")

0           data/interim/item_units.npy
1           data/interim/item_spend.npy
2          data/interim/chain_total.npy
3                data/interim/units.npy
4         data/interim/prices_daily.npy
5          data/interim/week_of_day.npy
6          data/interim/mask_reason.npy
7       data/interim/item_index.parquet
8     data/interim/series_index.parquet
9           data/interim/launch.parquet
10      data/interim/rejections.parquet
Name: archivos, dtype: str